In [2]:
import polars as pl
from datetime import timedelta

## Читаем файлы

In [3]:
data_path = '../data/raw/'
df_txns = pl.read_csv(f'{data_path}transactions.csv', try_parse_dates=True)
df_clients = pl.read_csv(f'{data_path}clients.csv', try_parse_dates=True)
df_kp = pl.read_csv(f'{data_path}kp.csv', try_parse_dates=True)
df_monthly= pl.read_csv(f'{data_path}monthly_client.csv', try_parse_dates=True)
df_nso = pl.read_csv(f'{data_path}monthly_nso.csv', try_parse_dates=True)

Находим точку отсчёта считаем активность за последние 2, 3 или 6 месяцев

In [4]:
max_date = df_txns.select(pl.col('txn_date').max()).item()

date_2m = max_date - timedelta(days=60)
date_3m = max_date - timedelta(days=90)
date_6m = max_date - timedelta(days=180)

## Группировка и подсчёт

In [5]:
txns_summary = df_txns.group_by('client_id').agg([
    pl.col('txn_date').min().alias('first_txn_date'),
    pl.col('txn_date').max().alias('last_txn_date'),
    (pl.col('txn_date') >= date_2m).sum().alias('txn_cnt_2m'),
    (pl.col('txn_date') >= date_3m).sum().alias('txn_cnt_3m'),
    (pl.col('txn_date') >= date_6m).sum().alias('txn_cnt_6m')
])

## Добавление recency (давность)

Мы вычисляем разницу в днях между крайней датой датасета и последней операцией клиента

Клиент перестал проводить операции 30-45 дней назад – это сигнал оттока

In [6]:
txns_summary = txns_summary.with_columns(
    (pl.lit(max_date) - pl.col('last_txn_date')).dt.total_days().alias('days_since_last_txn')
)

## Соединяем с профилем клиента

In [7]:
# profile = (
#     df_clients
#     .select(['client_id', 'registration_date'])
#     .join(txns_summary, on='client_id', how='left')
#     .with_columns([
#             pl.col('txn_cnt_2m').fill_null(0),
#             pl.col('txn_cnt_3m').fill_null(0),
#             pl.col('txn_cnt_6m').fill_null(0)
#         ])
# )


## Загрузка витрины и справочника отраслей

In [8]:
df_monthly = pl.read_csv(f'{data_path}monthly_client.csv', try_parse_dates=True)
df_kp = pl.read_csv(f'{data_path}kp.csv')

## Выделение последнего актуального состояния клиента

In [9]:
monthly_cols = [
    'client_id', 'report_date',
    'demand_balance_end', 'demand_balance_avg',
    'bal_cv', 'total_liabilities_end',
    'txn_amt', 'net_flow_amt', 'outflow_share', 'target_outflow_amt', 'uniq_ca_cnt',
    'has_credit', 'credit_util', 'salary_fund_amt'
]
latest_monthly = (
    df_monthly
    .select(monthly_cols)
    .sort('report_date')
    .group_by('client_id')
    .last()
)

## Сборка итогового расширенного профиля

In [10]:
profile = (
    df_clients
    .join(df_kp.select(['kp_id', 'kp_name']), on='kp_id', how='left')
    .join(latest_monthly, on='client_id', how='left')
    .join(txns_summary, on='client_id', how='left')
    .with_columns([
        pl.col('txn_cnt_2m').fill_null(0),
        pl.col('txn_cnt_3m').fill_null(0),
        pl.col('txn_cnt_6m').fill_null(0),
        pl.col('days_since_last_txn').fill_null(999),
        pl.col('has_credit').fill_null(0),
        pl.col('demand_balance_end').fill_null(0.0),
        pl.col('total_liabilities_end').fill_null(0.0)
    ])
)

## Проверка размерности

In [11]:
profile.shape

(3000, 27)

Ровно 3000 строк, как и клиентов, значит всё верно

## Вывод

In [14]:
profile

client_id,cpr_id,kp_id,segment_name,registration_date,has_salary_project,employee_cnt_start,kp_name,report_date,demand_balance_end,demand_balance_avg,bal_cv,total_liabilities_end,txn_amt,net_flow_amt,outflow_share,target_outflow_amt,uniq_ca_cnt,has_credit,credit_util,salary_fund_amt,first_txn_date,last_txn_date,txn_cnt_2m,txn_cnt_3m,txn_cnt_6m,days_since_last_txn
str,str,str,str,date,i64,i64,str,date,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,date,date,u32,u32,u32,i64
"""CL0000001""","""CPR000639""","""KP01""","""MASS""",2013-12-02,0,0,"""Добыча и сырье""",2026-05-31,2267723.7,2.0392e6,0.1379,3062701.7,3.5419e6,457139.5,0.4355,0.0,5,0,0.0,0.0,2024-01-02,2026-05-24,22,33,66,7
"""CL0000002""","""CPR000581""","""KP10""","""Alfa Only""",2000-02-26,1,129,"""АПК""",2026-05-31,113730.1,2.0545e6,0.3829,3.2553e6,7.2639e6,-3.8816e6,0.7672,0.0,9,1,0.2092,5.0853e6,2024-01-02,2026-05-26,22,33,65,5
"""CL0000003""","""CPR000075""","""KP08""","""MASS""",2002-07-31,1,97,"""Недвижимость""",2026-05-31,136735.57,2.2503e6,1.8075,136735.57,9.1729e6,-4.2272e6,0.7304,0.0,7,1,0.2071,5.7096e6,2024-01-01,2026-05-31,20,30,59,0
"""CL0000004""","""CPR000027""","""KP10""","""MASS""",2005-04-04,0,0,"""АПК""",2026-05-31,2.8268e6,2.5834e6,0.177,4.9137e6,3.1199e6,486679.78,0.422,0.0,5,1,0.5034,0.0,2024-01-01,2026-05-31,22,33,66,0
"""CL0000005""","""CPR000254""","""KP10""","""MASS""",2022-06-12,1,115,"""АПК""",2026-05-31,23021.21,300839.17,0.5058,276292.44,1.7004e6,-555635.93,0.6634,0.0,8,1,0.2034,1.0956e6,2024-01-07,2026-05-31,22,33,66,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CL0002996""","""CPR000296""","""KP10""","""MASS""",2013-06-19,1,147,"""АПК""",2026-05-31,70466.48,1.5172e6,2.6208,270304.22,4012228.3,-2.8935e6,0.8606,0.0,10,1,0.2617,3.2877e6,2024-01-04,2026-05-29,22,32,66,2
"""CL0002997""","""CPR000307""","""KP07""","""MASS""",2012-08-02,1,26,"""Финансы""",2026-05-31,84801.19,1124736.4,2.85,4.3155e6,6.2306e6,-2.0799e6,0.6669,0.0,7,0,0.0,2.9240e6,2024-01-01,2026-05-30,22,32,66,1
"""CL0002998""","""CPR000425""","""KP07""","""A-Club""",2012-11-08,1,237,"""Финансы""",2026-05-31,192055.14,2.8640e6,0.7121,192055.14,1.3478e7,-5.3439e6,0.6983,0.0,6,1,0.6256,7.8803e6,2024-01-01,2026-05-23,20,30,60,8
